In [2]:
import pandas as pd 
import numpy as np
from scipy import stats # for chi-sq and mann-whitney U test func
import matplotlib.pyplot as plt 
import seaborn as sns 
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns',50)
pd.set_option('display.float_format','{:.4f}'.format)


In [3]:
df=pd.read_parquet('../data/amex_clean.parquet')

print(f'Shape:{df.shape}')
print(f'Features:{df.shape[1]-1}')

Shape:(458913, 160)
Features:159


In [4]:
cat_cols = [col for col in ['B_30', 'B_38', 'D_114', 'D_116', 'D_117',
                              'D_120', 'D_126', 'D_63', 'D_64', 'D_66', 'D_68']
            if col in df.columns]

numeric_cols = [col for col in df.columns if col not in cat_cols + ['target']]

print(f'Categorical features: {len(cat_cols)}')
print(f'Numeric features: {len(numeric_cols)}')

Categorical features: 10
Numeric features: 149


In [5]:
chi2_results=[]

for col in cat_cols:
    contingency_table = pd.crosstab(df[col],df['target'])
    chi2_stat,p_value,dof,expected=stats.chi2_contingency(contingency_table)
    chi2_results.append({'feature':col,'chi2_stat':chi2_stat,'p_value':p_value})
    
chi2_df=pd.DataFrame(chi2_results).sort_values('chi2_stat',ascending=False)
chi2_df['significant']=chi2_df['p_value']<0.05

print('Chi-squared test results(categorical features vs target)')
print(chi2_df.to_string(index=False))

Chi-squared test results(categorical features vs target)
feature   chi2_stat  p_value  significant
   B_38 128079.1171   0.0000         True
   B_30  75874.9364   0.0000         True
  D_120  26113.8205   0.0000         True
   D_64  17390.3417   0.0000         True
   D_68  16763.0061   0.0000         True
  D_114  16550.4659   0.0000         True
  D_117  14289.7467   0.0000         True
   D_63   3636.3864   0.0000         True
  D_116   1396.6248   0.0000         True
  D_126   1391.9441   0.0000         True


In [12]:
def calculate_woe_iv_categorical(df, feature, target):
    df_temp = df[[feature, target]].copy()

    # each unique category value is its own bin — no qcut needed
    grouped = df_temp.groupby(feature, observed=True)[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']

    total_good = grouped['good'].sum()
    total_bad = grouped['bad'].sum()

    grouped['good_pct'] = grouped['good'] / total_good
    grouped['bad_pct'] = grouped['bad'] / total_bad

    grouped['good_pct'] = grouped['good_pct'].replace(0, 0.0001)
    grouped['bad_pct'] = grouped['bad_pct'].replace(0, 0.0001)

    grouped['woe'] = np.log(grouped['good_pct'] / grouped['bad_pct'])
    grouped['iv'] = (grouped['good_pct'] - grouped['bad_pct']) * grouped['woe']

    total_iv = grouped['iv'].sum()

    return grouped, total_iv


# run on all categorical features
cat_iv_summary = []

for col in cat_cols:
    try:
        _, iv = calculate_woe_iv_categorical(df, col, 'target')
        cat_iv_summary.append({'feature': col, 'iv': iv})
    except Exception as e:
        cat_iv_summary.append({'feature': col, 'iv': np.nan})

cat_iv_df = pd.DataFrame(cat_iv_summary).sort_values('iv', ascending=False).reset_index(drop=True)
cat_iv_df['strength'] = cat_iv_df['iv'].apply(iv_strength)

print('Categorical Features — IV Summary:')
print(cat_iv_df.to_string(index=False))

Categorical Features — IV Summary:
feature     iv   strength
   B_38 1.6506 Suspicious
   B_30 0.7511 Suspicious
  D_120 0.2633     Medium
   D_64 0.1978     Medium
   D_68 0.1847     Medium
  D_114 0.1834     Medium
  D_117 0.1712     Medium
   D_63 0.0449     Medium
  D_126 0.0153 Not useful
  D_116 0.0132 Not useful


In [6]:
#T-test/mann whitney for numeric
numerical_test_results=[]

for col in numeric_cols:
    group_0=df[df['target']==0][col]
    group_1=df[df['target']==1][col]
    
    stat,p_value=stats.mannwhitneyu(group_0,group_1,alternative='two-sided')
    numerical_test_results.append({'feature':col,'statistic':stat,'p_value':p_value})
    
numerical_test_df=pd.DataFrame(numerical_test_results).sort_values('p_value')
numerical_test_df['significant']=numerical_test_df['p_value']<0.05

print(f'Total numeric features tested:{len(numerical_test_df)}')
print(f'Statistically significant(p<0.005):{numerical_test_df["significant"].sum()}')
print()
print('Top 20 most significant numeric features:')
print(numerical_test_df.head(20).to_string(index=False))


Total numeric features tested:149
Statistically significant(p<0.005):145

Top 20 most significant numeric features:
feature        statistic  p_value  significant
    P_2 37020700011.5000   0.0000         True
   D_39 13006108287.5000   0.0000         True
    B_1  6651656973.5000   0.0000         True
    B_2 33922824818.5000   0.0000         True
    R_1 11918246385.0000   0.0000         True
    S_3 10267827603.5000   0.0000         True
   D_41 13345823433.5000   0.0000         True
    B_3  6503224075.5000   0.0000         True
   D_43 13108905117.5000   0.0000         True
   D_44  6646934729.0000   0.0000         True
    B_4  7250732109.5000   0.0000         True
   D_45 28541521792.0000   0.0000         True
    B_5 23409651704.5000   0.0000         True
    R_2 15098309039.0000   0.0000         True
   D_46 14104909466.0000   0.0000         True
   D_47 28339032033.0000   0.0000         True
   D_48  5175675721.0000   0.0000         True
    B_6 33798969465.5000   0.0000     

In [7]:
#WOE/IV
def calculate_woe_iv(df,feature,target,bins=10):
    df_temp=df[[feature,target]].copy()
    df_temp['bin']=pd.qcut(df_temp[feature],q=bins,duplicates='drop')
    
    grouped=df_temp.groupby('bin',observed=True)[target].agg(['count','sum'])
    grouped.columns=['total','bad']
    grouped['good']=grouped['total']-grouped['bad']
    
    total_good=grouped['good'].sum()
    total_bad=grouped['bad'].sum()
    
    grouped['good_pct']=grouped['good']/total_good
    grouped['bad_pct']=grouped['bad']/total_bad
    
    grouped['good_pct']=grouped['good_pct'].replace(0,0.0001)
    grouped['bad_pct']=grouped['bad_pct'].replace(0,0.0001)
    
    grouped['woe']=np.log(grouped['good_pct']/grouped['bad_pct'])
    grouped['iv']=(grouped['good_pct']-grouped['bad_pct'])*grouped['woe']
    
    total_iv=grouped['iv'].sum()
    return grouped,total_iv

In [8]:
iv_summary=[]
for col in numeric_cols:
    try:
        _,iv=calculate_woe_iv(df,col,'target',bins=10)
        iv_summary.append({'feature':col,'iv':iv})
    except Exception as e:
        iv_summary.append({'feature':col,'iv':np.nan})
iv_df=pd.DataFrame(iv_summary).sort_values('iv',ascending=False).reset_index(drop=True)

print(f'IV calculated for {iv_df["iv"].notna().sum()} out of {len(iv_df)} numeric features')

print('Top 20 features by info value:')
print(iv_df.head(20).to_string(index=False))



IV calculated for 149 out of 149 numeric features
Top 20 features by info value:
feature     iv
    P_2 3.6611
   D_48 2.3690
   B_18 2.1443
    B_7 2.0903
   B_23 2.0447
   D_61 2.0279
   B_10 2.0068
   D_75 1.9939
    B_9 1.9555
    B_2 1.8769
    B_6 1.8621
    B_3 1.8425
   D_44 1.8121
    B_1 1.7993
   B_37 1.7740
   D_55 1.7631
   B_11 1.7117
   D_62 1.6034
    B_4 1.5619
   B_40 1.5367


In [9]:
def iv_strength(iv):
    if iv<0.02:
        return 'Not useful'
    elif iv<0.1:
        return 'Medium'
    elif iv<0.3:
        return 'Medium'
    elif iv<0.5:
        return 'Strong'
    else:
        return 'Suspicious'

iv_df['strength']=iv_df['iv'].apply(iv_strength)

print('Feature count by IV strength category:')
print(iv_df['strength'].value_counts())

Feature count by IV strength category:
strength
Medium        56
Suspicious    51
Not useful    28
Strong        14
Name: count, dtype: int64


In [13]:
# numeric — drop IV < 0.02
selected_numeric = iv_df[iv_df['iv'] >= 0.02]['feature'].tolist()

# categorical — drop IV < 0.02
selected_cat = cat_iv_df[cat_iv_df['iv'] >= 0.02]['feature'].tolist()

# combine
final_features = selected_numeric + selected_cat + ['target']
df_selected = df[final_features]

print(f'Selected numeric features: {len(selected_numeric)}')
print(f'Selected categorical features: {len(selected_cat)}')
print(f'Total features selected: {len(selected_numeric) + len(selected_cat)}')
print(f'Final dataset shape: {df_selected.shape}')

Selected numeric features: 121
Selected categorical features: 8
Total features selected: 129
Final dataset shape: (458913, 130)


In [14]:
OUTPUT_PATH = r'..\data\amex_selected_features.parquet'
IV_TABLE_PATH = r'..\outputs\iv_table.csv'
CAT_IV_TABLE_PATH = r'..\outputs\cat_iv_table.csv'

df_selected.to_parquet(OUTPUT_PATH, index=False)
iv_df.to_csv(IV_TABLE_PATH, index=False)
cat_iv_df.to_csv(CAT_IV_TABLE_PATH, index=False)

print(f'Selected features dataset saved: {df_selected.shape}')
print(f'Numeric IV table saved to outputs/')
print(f'Categorical IV table saved to outputs/')

Selected features dataset saved: (458913, 130)
Numeric IV table saved to outputs/
Categorical IV table saved to outputs/
